# WeedBot — YOLOv8-nano-seg Fine-tuning Notebook

This notebook fine-tunes `yolov8n-seg` on labeled feedback exported from the WeedBot Streamlit app.

## Workflow
1. Upload `training_export.zip` to your Google Drive (any folder you like).
2. Update the `DRIVE_ZIP_PATH` constant in **Cell 2** to point to your zip.
3. Run all cells top-to-bottom.
4. The trained weights (`best.pt`) are saved to `WeedBot/weights/` in your Drive.
5. Download `best.pt`, place it anywhere on your local machine, and update
   `model_weights_path` in your local `config.yaml` to point to it.
6. Restart the Streamlit app — it will load the new weights automatically.

**Runtime:** Use `Runtime → Change runtime type → T4 GPU` for faster training.
CPU-only training works but is significantly slower.

In [7]:
# ── Cell 1: Configuration ────────────────────────────────────────────────────
# Path inside your Google Drive where you uploaded the training_export.zip.
# Example: '/content/drive/MyDrive/weedbot/training_export.zip'
DRIVE_ZIP_PATH = '/content/drive/My Drive/training_export.zip'

# Where to save the trained weights inside Drive.
DRIVE_WEIGHTS_DIR = '/content/drive/My Drive/weights'

# Training hyperparameters — adjust as needed.
EPOCHS      = 20
IMAGE_SIZE  = 640
BATCH_SIZE  = 4   # reduce to 4 if you run out of GPU memory

# Class mapping used during export.
# Must match the label_map in your local config.yaml.
# 0 = crop, 1 = weed  (soil detections are not currently used for training)
CLASS_NAMES = ['crop', 'weed']

print('Configuration loaded.')

Configuration loaded.


In [4]:
# ── Cell 2: Mount Google Drive ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted.


In [5]:
# ── Cell 3: Install dependencies ─────────────────────────────────────────────
!pip install -q 'ultralytics>=8.0'
import ultralytics
print(f'ultralytics {ultralytics.__version__} ready.')

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
ultralytics 8.4.202 ready.


In [8]:
# ── Cell 4: Unzip the training export ────────────────────────────────────────
import zipfile, shutil
from pathlib import Path

EXPORT_DIR = Path('/content/training_export')
if EXPORT_DIR.exists():
    shutil.rmtree(EXPORT_DIR)
EXPORT_DIR.mkdir()

with zipfile.ZipFile(DRIVE_ZIP_PATH, 'r') as zf:
    zf.extractall(EXPORT_DIR)

labels_csv = EXPORT_DIR / 'labels.csv'
images_dir = EXPORT_DIR / 'images'
assert labels_csv.exists(), f'labels.csv not found in {EXPORT_DIR}'
assert images_dir.exists(), f'images/ directory not found in {EXPORT_DIR}'

import csv
with labels_csv.open() as fh:
    rows = list(csv.DictReader(fh))
print(f'Found {len(rows)} labeled samples in labels.csv.')

Found 23 labeled samples in labels.csv.


In [9]:
# ── Cell 5: Validate dataset format (dry-run smoke test) ─────────────────────
# This cell verifies the export structure without starting any training.
# It should always pass; if it fails, check that your training_export.zip
# was generated by the latest version of the WeedBot app.

import csv
from pathlib import Path

REQUIRED_FIELDS = {'patch_path', 'image_id', 'proposal_id',
                   'model_decision', 'pull_decision_correct', 'annotated_at'}

labels_csv = EXPORT_DIR / 'labels.csv'
with labels_csv.open() as fh:
    reader = csv.DictReader(fh)
    assert REQUIRED_FIELDS.issubset(set(reader.fieldnames or [])), \
        f'Missing columns: {REQUIRED_FIELDS - set(reader.fieldnames or [])}'
    rows = list(reader)

missing_images = []
for row in rows:
    patch = EXPORT_DIR / row['patch_path']
    if not patch.exists():
        missing_images.append(row['patch_path'])

if missing_images:
    print(f'WARNING: {len(missing_images)} patch image(s) not found — they will be skipped.')
    print('  First missing:', missing_images[0])
else:
    print(f'All {len(rows)} patch images present. Dataset looks good.')

label_counts = {}
for row in rows:
    val = row['pull_decision_correct']
    label_counts[val] = label_counts.get(val, 0) + 1
print('Label distribution:', label_counts)

  First missing: images\demo_0001_demo_0001_weed_002_bbox_clearance_v1.png
Label distribution: {'correct': 7, 'incorrect': 16}


In [19]:
# ── Cell 6: Convert to YOLOv8 segmentation dataset format ────────────────────
#
# YOLOv8-seg expects:
#   dataset_root/
#     images/train/   *.png
#     images/val/     *.png
#     labels/train/   *.txt   (one line per instance: class cx cy w h [poly])
#     labels/val/     *.txt
#     dataset.yaml
#
# Because our patches are already cropped to the weed + context, each patch
# contains exactly one primary annotation: the weed centred in the image.
# The model_decision (attempt/skip) combined with pull_decision_correct
# determines the ground-truth class:
#   - correct + attempt  OR  incorrect + skip  → this IS a weed  (class 1)
#   - correct + skip     OR  incorrect + attempt → this is crop/non-weed (class 0)
#
# The bounding annotation is a full-image box (the whole patch = the object)
# because patches are cropped around the target.

import csv, shutil, random
from pathlib import Path

YOLO_ROOT = Path('/content/yolo_dataset')
for split in ('train', 'val'):
    (YOLO_ROOT / 'images' / split).mkdir(parents=True, exist_ok=True)
    (YOLO_ROOT / 'labels' / split).mkdir(parents=True, exist_ok=True)

with (EXPORT_DIR / 'labels.csv').open() as fh:
    all_rows = [r for r in csv.DictReader(fh)
                if r['pull_decision_correct'] in ('correct', 'incorrect')]

random.seed(42)
random.shuffle(all_rows)
split_idx = max(1, int(len(all_rows) * 0.8))
splits = {'train': all_rows[:split_idx], 'val': all_rows[split_idx:]}

def row_to_class(row):
    """Return 1 (weed) or 0 (crop/non-weed) based on decision × feedback."""
    decision = row['model_decision']        # 'attempt' or 'skip'
    correct  = row['pull_decision_correct'] # 'correct' or 'incorrect'
    is_weed = (
        (decision == 'attempt' and correct == 'correct') or
        (decision == 'skip'    and correct == 'incorrect')
    )
    return 1 if is_weed else 0

counts = {'train': 0, 'val': 0}
for split, rows in splits.items():
    for row in rows:
        src = EXPORT_DIR / row['patch_path']
        if not src.exists():
            continue
        stem = src.stem
        dst_img   = YOLO_ROOT / 'images' / split / src.name
        dst_label = YOLO_ROOT / 'labels' / split / (stem + '.txt')
        shutil.copy2(src, dst_img)
        cls = row_to_class(row)
        # YOLOv8-seg label format: class cx cy w h  x1 y1 x2 y2 x3 y3 x4 y4
        # The patch is cropped around the target, so the whole image is the
        # object.  A tight full-image rectangle works as the polygon.
        # All coordinates are normalised 0-1.
        poly = '0.0 0.0  1.0 0.0  1.0 1.0  0.0 1.0'
        dst_label.write_text(f'{cls} 0.5 0.5 1.0 1.0 {poly}\n')
        counts[split] += 1

print(f'Dataset built — train: {counts["train"]}, val: {counts["val"]}')

dataset_yaml = YOLO_ROOT / 'dataset.yaml'
dataset_yaml.write_text(
    f"path: {YOLO_ROOT}\n"
    f"train: images/train\n"
    f"val:   images/val\n"
    f"nc: {len(CLASS_NAMES)}\n"
    f"names: {CLASS_NAMES}\n"
)
print('dataset.yaml written.')

Dataset built — train: 18, val: 5
dataset.yaml written.


In [20]:
# ── Cell 7: Fine-tune YOLOv8-nano-seg ────────────────────────────────────────
from ultralytics import YOLO

model = YOLO('yolov8n-seg.pt')  # start from pretrained COCO weights

results = model.train(
    data=str(YOLO_ROOT / 'dataset.yaml'),
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    project='/content/runs',
    name='weedbot_finetune',
    exist_ok=True,
    verbose=True,
)
print('Training complete.')

Ultralytics 8.4.202 🚀 Python-3.13.18 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo_dataset/dataset.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=weedbot_finetune, nbs=64, nms=None, opset=None, opti

In [21]:
# ── Cell 8: Save best.pt to Google Drive ─────────────────────────────────────
import shutil
from pathlib import Path

best_pt = Path('/content/runs/weedbot_finetune/weights/best.pt')
assert best_pt.exists(), f'Weights not found at {best_pt}. Did training complete?'

drive_weights = Path(DRIVE_WEIGHTS_DIR)
drive_weights.mkdir(parents=True, exist_ok=True)
dest = drive_weights / 'best.pt'
shutil.copy2(best_pt, dest)
print(f'Saved weights to {dest}')

Saved weights to /content/drive/My Drive/weights/best.pt


In [22]:
# ── Cell 9: Next steps ────────────────────────────────────────────────────────
print("""
╔══════════════════════════════════════════════════════════════════════╗
║  Training complete!  Here is how to use your new weights locally.   ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  1. In Google Drive, download:                                       ║
║       WeedBot/weights/best.pt                                        ║
║                                                                      ║
║  2. Place best.pt anywhere on your local machine, e.g.:             ║
║       outputs/weights/best.pt                                        ║
║                                                                      ║
║  3. Edit config.yaml in your project root:                          ║
║       backend: model                                                 ║
║       model_weights_path: outputs/weights/best.pt                   ║
║                                                                      ║
║  4. Also update label_map to match your trained class names:        ║
║       label_map:                                                     ║
║         crop: crop                                                   ║
║         weed: weed                                                   ║
║                                                                      ║
║  5. Restart the Streamlit app:                                       ║
║       streamlit run app.py                                           ║
║                                                                      ║
║  The app will load the new weights and run inference on each image. ║
╚══════════════════════════════════════════════════════════════════════╝
""")


╔══════════════════════════════════════════════════════════════════════╗
║  Training complete!  Here is how to use your new weights locally.   ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  1. In Google Drive, download:                                       ║
║       WeedBot/weights/best.pt                                        ║
║                                                                      ║
║  2. Place best.pt anywhere on your local machine, e.g.:             ║
║       outputs/weights/best.pt                                        ║
║                                                                      ║
║  3. Edit config.yaml in your project root:                          ║
║       backend: model                                                 ║
║       model_weights_path: outputs/weights/best.pt                   ║
║                                                     